# inspect_dataset.py

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd

dataset = Path("data set/tomato-dataset")

extensions = {".jpg", ".jpeg", ".png", ".webp"}
results = []

for split in ["train", "val"]:
    folder = dataset / split

    for disease in sorted(folder.iterdir()):
        if not disease.is_dir():
            continue

        images = [
            file for file in disease.rglob("*")
            if file.suffix.lower() in extensions
        ]

        results.append({
            "Split": split,
            "Disease": disease.name,
            "Images": len(images)
        })

df = pd.DataFrame(results)

print("\nDataset Information:")
print(df.to_string(index=False))

print("\nTotal Images:", df["Images"].sum())
print("Number of Classes:", df["Disease"].nunique())

print("\nChecking Images...")

corrupted = []

for split in ["train", "val"]:
    folder = dataset / split

    for file in folder.rglob("*"):
        if file.suffix.lower() in extensions:
            try:
                with Image.open(file) as img:
                    img.verify()
            except Exception:
                corrupted.append(str(file))

print("Corrupted Images:", len(corrupted))

for file in corrupted[:10]:
    print(file)

# check_duplicates.py

In [ ]:
from pathlib import Path
import hashlib

dataset = Path("data set/tomato-dataset")

def get_hashes(split):
    hashes = set()

    for file in (dataset / split).rglob("*"):
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            with open(file, "rb") as f:
                image_hash = hashlib.md5(f.read()).hexdigest()
                hashes.add(image_hash)

    return hashes

train_hashes = get_hashes("train")
val_hashes = get_hashes("val")

duplicates = train_hashes.intersection(val_hashes)

print("Train Unique Images:", len(train_hashes))
print("Validation Unique Images:", len(val_hashes))
print("Duplicate Images:", len(duplicates))

# prepare_dataset.py

In [ ]:
from pathlib import Path
import random
import shutil

random.seed(42)

source = Path("data set/tomato-dataset")
destination = Path("prepared_dataset")

extensions = {".jpg", ".jpeg", ".png", ".webp"}

if destination.exists():
    raise SystemExit("prepared_dataset already exists. No files changed.")

for class_folder in sorted((source / "train").iterdir()):
    if not class_folder.is_dir():
        continue

    images = [
        f for f in class_folder.iterdir()
        if f.suffix.lower() in extensions
    ]

    random.shuffle(images)

    split_index = int(len(images) * 0.9)

    train_images = images[:split_index]
    val_images = images[split_index:]

    for split, files in [
        ("train", train_images),
        ("val", val_images)
    ]:
        target = destination / split / class_folder.name
        target.mkdir(parents=True, exist_ok=True)

        for image in files:
            shutil.copy2(image, target / image.name)

    print(class_folder.name, len(train_images), len(val_images))

# Original validation becomes the final test set
shutil.copytree(source / "val", destination / "test")

print("\nDataset preparation completed!")